# 09 · Claude's features: preparing a board briefing

**Goal:** use five of Claude's features on one realistic job: preparing a board briefing for Halvora Systems, the fictional company in `data/halvora_report.md`.

| | |
|---|---|
| **You will learn** | Thinking and effort, reading images, reading PDFs, citations, and prompt caching |
| **Prerequisites** | [Notebook 02](02_structured_outputs.ipynb), for structured outputs |
| **Run time** | About 3 minutes |
| **Cost** | Under $0.30 expected. This notebook stops itself at $0.50 |
| **Paid cells** | Most cells in sections 1 to 6. Building the chart and the PDF, and counting tokens, are free |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

| Section | Feature | What you do with it |
|---|---|---|
| 1 | Thinking | Work out the order the division heads present in |
| 2 | Images | Have Claude read a KPI chart, and check its numbers |
| 3 | PDFs | Turn the report into a PDF and ask about it |
| 4 | Citations | Make every claim point to its exact source |
| 5 | Prompt caching | Ask many questions about one PDF without paying full price each time |
| 6 | All together | Write the cited board brief |

PDFs are the most expensive part of this notebook: each PDF request sends about 13,000 tokens, which is why caching gets a whole section.

## Setup

The usual setup, with one change: `track()` also prices **cache writes and cache reads**, which section 5 uses. Run the cells from top to bottom.

In [ ]:
import base64
import itertools
import os
import re
import time
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

ASSETS = Path("outputs/09_claude_features")   # chart and PDF go here (not committed)
ASSETS.mkdir(parents=True, exist_ok=True)
report_text = Path("data/halvora_report.md").read_text(encoding="utf-8")

print("Ready. Using model:", MODEL, "| report:", f"{len(report_text):,} characters")

In [ ]:
INPUT_PRICE = 2.00          # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00        # dollars per 1 million output tokens (Sonnet 5.5)
CACHE_WRITE_PRICE = 2.50    # dollars per 1 million tokens written to the 5-minute cache
CACHE_READ_PRICE = 0.10     # dollars per 1 million tokens read from the cache
BUDGET_USD = 0.50           # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    usage = response.usage
    written = usage.cache_creation_input_tokens or 0
    read = usage.cache_read_input_tokens or 0
    cost = (usage.input_tokens * INPUT_PRICE + written * CACHE_WRITE_PRICE
            + read * CACHE_READ_PRICE + usage.output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    cache_part = f" cache_write={written:,} cache_read={read:,}" if written or read else ""
    print(f"[input={usage.input_tokens:,}{cache_part} output={usage.output_tokens:,} "
          f"cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def answer_text(response):
    """Join ALL text blocks. Citations split an answer into many small text blocks, and
    thinking blocks may come first, so content[0].text is not enough."""
    return "".join(block.text for block in response.content if block.type == "text").strip()

## 1. Thinking: who presents when? (paid)

Seven division heads present to the board, and the chair has sent rules for the order. The answer goes straight into the agenda, so we ask for **only the names**.

**Thinking** gives Claude room to reason **before** it answers, in separate `thinking` blocks, so the visible answer can stay short. On Sonnet 5.5 you control it with two settings:

| Setting | What it does |
|---|---|
| `thinking={"type": "adaptive"}` | Claude decides whether and how much to think. **This is the default** |
| `output_config={"effort": ...}` | How hard to try: `low`, `medium`, `high` (the default), `xhigh` or `max` |
| `thinking={"type": "between_tools"}` | The lowest setting: no extended thinking. It allows effort up to `high`, and no other fields |
| `"display": "summarized"` | Return a readable summary of the thinking. Without it, thinking blocks come back empty, but **you're billed for the thinking either way** |

> **Different from the course:** the course sets a fixed thinking budget with `budget_tokens`. Sonnet 5.5 rejects that with a 400, and also rejects `{"type": "disabled"}`. Effort replaced the budget, and `between_tools` is the way to turn thinking off.

In notebook 00, `low` and `high` effort cost the same on an easy puzzle, because Claude didn't need to think. This puzzle is harder: nine rules and seven people. First, we find the right answer ourselves by checking every possible order, which is free.

In [ ]:
HEADS = ["Finance", "Engineering", "Research", "Legal", "HR", "Security", "Marketing"]

AGENDA_RULES = """Seven division heads present to the board, one after another:
Finance, Engineering, Research, Legal, HR, Security, Marketing.
1. Security presents immediately after Engineering.
2. Finance presents before Research but after Legal.
3. HR is not first and not last.
4. Research presents fourth or sixth.
5. Legal does not present next to HR.
6. Engineering presents after Finance.
7. Research does not present immediately after HR.
8. At least two other people present between Marketing and Finance.
9. Marketing presents after HR."""


def follows_rules(order):
    pos = {name: i for i, name in enumerate(order)}   # 0 = first
    return (pos["Security"] == pos["Engineering"] + 1
            and pos["Legal"] < pos["Finance"] < pos["Research"]
            and pos["HR"] not in (0, 6)
            and pos["Research"] in (3, 5)
            and abs(pos["Legal"] - pos["HR"]) != 1
            and pos["Engineering"] > pos["Finance"]
            and pos["Research"] != pos["HR"] + 1
            and abs(pos["Marketing"] - pos["Finance"]) >= 3
            and pos["Marketing"] > pos["HR"])


solutions = [order for order in itertools.permutations(HEADS) if follows_rules(order)]
print(f"Orders that satisfy all 9 rules, out of 5,040: {len(solutions)}")
print("The answer:", " -> ".join(solutions[0]))

assert len(solutions) == 1

Now we ask Claude three ways: with thinking off, with thinking at `low` effort, and at `high` effort. Each runs once, to keep the cost down, so treat the results as one sample, not a verdict.

In [ ]:
AGENDA_PROMPT = AGENDA_RULES + "\nReply with ONLY the seven names in order, comma-separated. No explanation."

SETTINGS = {
    "off (between_tools)": {"thinking": {"type": "between_tools"}, "output_config": {"effort": "low"}},
    "adaptive, low effort": {"thinking": {"type": "adaptive", "display": "summarized"}, "output_config": {"effort": "low"}},
    "adaptive, high effort": {"thinking": {"type": "adaptive", "display": "summarized"}, "output_config": {"effort": "high"}},
}

agenda_runs = {}
for label, setting in SETTINGS.items():
    start = time.perf_counter()
    response = track(client.messages.create(
        model=MODEL,
        max_tokens=8000,
        messages=[{"role": "user", "content": AGENDA_PROMPT}],
        **setting,
    ))
    visible = answer_text(response)
    names = tuple(re.findall("|".join(HEADS), visible.splitlines()[-1] if visible else ""))
    details = response.usage.output_tokens_details
    agenda_runs[label] = {
        "response": response,
        "correct": names == solutions[0],
        "only_names": len(visible.splitlines()) == 1,
        "visible_chars": len(visible),
        "thinking_tokens": details.thinking_tokens if details and details.thinking_tokens is not None else "n/a",
        "output_tokens": response.usage.output_tokens,
        "seconds": time.perf_counter() - start,
    }

print(f"\n{'setting':<24}{'correct':>8}{'only names':>12}{'answer chars':>14}{'thinking tok':>14}{'output tok':>12}{'time':>7}")
for label, r in agenda_runs.items():
    print(f"{label:<24}{'yes' if r['correct'] else 'no':>8}{'yes' if r['only_names'] else 'no':>12}"
          f"{r['visible_chars']:>14}{r['thinking_tokens']!s:>14}{r['output_tokens']:>12}{r['seconds']:>6.1f}s")

In [ ]:
high = agenda_runs["adaptive, high effort"]["response"]
off = agenda_runs["off (between_tools)"]["response"]

print("Thinking OFF, the visible answer (first 400 characters):")
print("   " + answer_text(off)[:400].replace("\n", "\n   "))

print("\nHigh effort, block types:", [block.type for block in high.content])
for block in high.content:
    if block.type == "thinking" and block.thinking:
        print("Thinking summary (first 400 characters):", block.thinking[:400].replace("\n", " "), "...")
print("Visible answer:", answer_text(high))

**What to notice**

- **Read `correct` and `only names` together.** With no thinking, Claude has nowhere to reason except the answer itself, so it may write out its working and break the format you asked for, or guess.
- **`thinking tok` and `output tok` are the price.** Thinking is billed as output, even though you only see a summary. High effort usually thinks longest.
- One run each is a small sample. Run the cell again and the numbers will move.
- **Continuing a conversation?** Send the assistant's thinking blocks back **unchanged**, by appending the whole `response.content`, as in notebook 01.

## 2. Images: can Claude read your chart? (paid)

Engineering sent a KPI chart for the briefing. Before it goes to the board, check that Claude reads it correctly.

**How images are sent:** an `image` content block, placed **before** your question in the same user message. The source can be base64 data (used here), a URL, or a `file_id` from the Files API. PNG, JPEG, GIF and WebP all work.

**Cost:** Claude reads an image in 28×28-pixel patches, and each patch is one token, so an image costs **⌈width ÷ 28⌉ × ⌈height ÷ 28⌉** tokens. Sonnet 5.5 reads images at high resolution: up to 2,576 pixels on the long edge, or 4,784 tokens. Larger images are scaled down automatically, so sending a huge screenshot only adds upload time.

> **Different from the course:** the course estimates width × height ÷ 750. The current docs use the 28-pixel patch rule above.

First we draw the chart. That's free.

In [ ]:
import matplotlib
matplotlib.use("Agg")                     # draw straight to a file
import matplotlib.pyplot as plt
from IPython.display import Image, display

KPIS = {                                  # metric: (2022, 2023), the TRUE values from the report
    "Customer-facing outages": (19, 7),
    "Avg recovery time (min)": (52, 18),
    "Bugs found before release (%)": (61, 74),
}
CHART_PATH = ASSETS / "engineering_kpis.png"

fig, ax = plt.subplots(figsize=(7.5, 4))
x = range(len(KPIS))
bars_2022 = ax.bar([i - 0.2 for i in x], [v[0] for v in KPIS.values()], width=0.4, label="2022", color="#9aa5b1")
bars_2023 = ax.bar([i + 0.2 for i in x], [v[1] for v in KPIS.values()], width=0.4, label="2023", color="#2f6fdb")
ax.bar_label(bars_2022, padding=2)
ax.bar_label(bars_2023, padding=2)
ax.set_xticks(list(x), list(KPIS), fontsize=9)
ax.set_title("Halvora Engineering KPIs, 2022 vs 2023")
ax.legend()
fig.tight_layout()
fig.savefig(CHART_PATH, dpi=110)
plt.close(fig)

display(Image(CHART_PATH))

Now we send it, and get the values back as a structured object (notebook 02), so code can check every number.

In [ ]:
import math

from PIL import Image as PILImage


class KPIReading(BaseModel):
    metric: str
    value_2022: float
    value_2023: float


class ChartReading(BaseModel):
    title: str
    kpis: list[KPIReading]
    biggest_relative_improvement: str


chart_b64 = base64.standard_b64encode(CHART_PATH.read_bytes()).decode("utf-8")
image_block = {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": chart_b64}}

response = track(client.messages.parse(
    model=MODEL,
    max_tokens=2000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": [
        image_block,                                          # the image FIRST, then the question
        {"type": "text", "text": "Read this chart. Give its title, every KPI with its 2022 and 2023 values, "
                                 "and which KPI improved the most in relative terms."},
    ]}],
    output_format=ChartReading,
))
reading = response.parsed_output

width, height = PILImage.open(CHART_PATH).size
image_tokens = math.ceil(width / 28) * math.ceil(height / 28)   # one token per 28x28-pixel patch
print(f"Chart is {width}x{height} pixels, so about {image_tokens:,} image tokens\n")
print("Title:", reading.title)

# Check every value against the truth. Claude may word the metric names differently, so match on a keyword.
keywords = {"Customer-facing outages": "outage", "Avg recovery time (min)": "recovery",
            "Bugs found before release (%)": "release"}
for metric, (true_2022, true_2023) in KPIS.items():
    match = next((k for k in reading.kpis if keywords[metric] in k.metric.lower()), None)
    if match is None:
        print(f"   MISSING  {metric}")
        continue
    ok = (match.value_2022, match.value_2023) == (true_2022, true_2023)
    print(f"   {'correct' if ok else 'WRONG  '}  {metric:<31} Claude: {match.value_2022:g} -> {match.value_2023:g}"
          f"   true: {true_2022} -> {true_2023}")
print("\nBiggest relative improvement:", reading.biggest_relative_improvement)

assert reading.kpis

**What to notice**

- **Always check numbers Claude reads from an image** when they matter. Here the bars carry value labels, which makes reading easy.
- Outages fell 63%, recovery time 65%, and the share of bugs found before release rose 21%. Did Claude pick recovery time as the biggest improvement?

## 3. PDFs: the report as the board will see it (free, then paid)

The board gets the report as a PDF, with the KPI chart as an appendix. First we build that PDF, which is free.

**How PDFs work:** a `document` content block. Claude receives **two things from every page**: the extracted **text**, and an **image** of the page. That's how it can read charts, tables and layout, and it's also why a PDF costs more than the same text.

Limits: 32 MB per request, and up to 600 pages on 1M-context models such as Sonnet 5.5.

In [ ]:
from fpdf import FPDF

PDF_PATH = ASSETS / "halvora_board_report.pdf"


def latin1(text):
    """fpdf2's built-in fonts only know Latin-1 characters, so swap the rest."""
    for fancy, plain in {"\u2014": "-", "\u2013": "-", "\u2019": "'", "\u2018": "'",
                         "\u201c": '"', "\u201d": '"', "\u2026": "...", "\u00d7": "x"}.items():
        text = text.replace(fancy, plain)
    return text.replace("**", "").replace("*", "").encode("latin-1", "replace").decode("latin-1")


pdf = FPDF()
pdf.set_auto_page_break(True, margin=15)
pdf.add_page()
for line in report_text.splitlines():
    if line.startswith("# "):
        pdf.set_font("Helvetica", "B", 16)
        pdf.multi_cell(0, 9, latin1(line[2:]), new_x="LMARGIN", new_y="NEXT")
        pdf.ln(2)
    elif line.startswith("## "):
        pdf.ln(3)
        pdf.set_font("Helvetica", "B", 13)
        pdf.multi_cell(0, 7, latin1(line[3:]), new_x="LMARGIN", new_y="NEXT")
        pdf.ln(1)
    elif line.strip():
        pdf.set_font("Helvetica", "", 10.5)
        pdf.multi_cell(0, 5.5, latin1(line), new_x="LMARGIN", new_y="NEXT")
    else:
        pdf.ln(2)

pdf.add_page()                                            # the appendix: the chart only, no numbers in text
pdf.set_font("Helvetica", "B", 13)
pdf.multi_cell(0, 7, "Appendix A: Engineering KPI chart", new_x="LMARGIN", new_y="NEXT")
pdf.image(str(CHART_PATH), w=180)
pdf.output(str(PDF_PATH))

pdf_b64 = base64.standard_b64encode(PDF_PATH.read_bytes()).decode("utf-8")
print(f"Built {PDF_PATH.name}: {pdf.page_no()} pages, {PDF_PATH.stat().st_size:,} bytes")


def pdf_block(citations=False, cache=False):
    """The board PDF as a document block. Sections 4 to 6 switch on citations and caching."""
    block = {"type": "document", "title": "Halvora Annual Review 2023",
             "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_b64}}
    if citations:
        block["citations"] = {"enabled": True}
    if cache:
        block["cache_control"] = {"type": "ephemeral"}
    return block

**How much does a PDF cost?** `count_tokens` is free, so we compare the PDF with the same report sent as plain text.

In [ ]:
question = "Look at the chart in Appendix A. What colour are the 2022 bars and the 2023 bars?"

pdf_tokens = client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": [
    pdf_block(), {"type": "text", "text": question}]}]).input_tokens
text_tokens = client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content":
    report_text + "\n\n" + question}]).input_tokens

print(f"As a PDF   : {pdf_tokens:,} input tokens")
print(f"As text    : {text_tokens:,} input tokens")
print(f"The PDF costs {pdf_tokens / text_tokens:.1f}x as much to send.")

assert pdf_tokens > text_tokens

Now the paid question. The bar colours appear **only** in the chart image inside the PDF: no sentence in the report mentions them. So a correct answer shows Claude really looked at the page image.

The PDF block is marked for caching here, so section 5 can reuse it.

In [ ]:
response = track(client.messages.create(
    model=MODEL,
    max_tokens=500,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": [pdf_block(cache=True), {"type": "text", "text": question}]}],
))
print("Q:", question)
print("A:", answer_text(response))

**What to notice**

- The answer should be grey for 2022 and blue for 2023. Those colours exist only in the image of the appendix page.
- When you test this yourself, ask about something that is **really** only in an image. The KPI numbers also appear in the report's text, so a question about them wouldn't prove Claude read the chart.
- **If you only need the words, send text.** Send a PDF when layout, tables or charts matter.

## 4. Citations: every claim points to its source (paid)

A board member will ask *"where does it say that?"* With citations switched on, Claude's answer comes back as several text blocks, and the ones backed by the document carry a `citations` list with the **exact quoted text** and where it came from.

| Document type | Citation type | Location fields |
|---|---|---|
| PDF | `page_location` | `start_page_number`, `end_page_number`, counting from 1 |
| Plain text | `char_location` | `start_char_index`, `end_char_index`, counting from 0 |

Both end values are **exclusive**, like Python slices. The quoted `cited_text` is free: it doesn't count as output tokens.

Turn citations on with `"citations": {"enabled": True}` on the document block. **They can't be combined with structured outputs**: that request returns a 400.

In [ ]:
from IPython.display import Markdown


def render_cited_answer(response):
    """Show the answer with [n] markers, and a numbered source list under it."""
    answer, sources = "", []
    for block in response.content:
        if block.type != "text":
            continue                                       # skip thinking blocks
        answer += block.text
        for citation in block.citations or []:
            sources.append(citation)
            answer += f" **[{len(sources)}]**"
    lines = ["### Answer", answer.strip(), "", "### Sources"]
    for n, c in enumerate(sources, 1):
        if c.type == "page_location":
            last_page = c.end_page_number - 1              # the end is exclusive
            where = f"page {c.start_page_number}" + (f"-{last_page}" if last_page > c.start_page_number else "")
        else:
            where = f"characters {c.start_char_index}-{c.end_char_index}"
        lines.append(f'{n}. *{c.document_title}*, {where}: "{" ".join(c.cited_text.split())}"')
    display(Markdown("\n".join(lines)))
    return sources


question = "What was incident INC-2023-Q4-011, and how did the engineering team help resolve it?"

response = track(client.messages.create(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": [pdf_block(citations=True), {"type": "text", "text": question}]}],
))
pdf_sources = render_cited_answer(response)

assert pdf_sources, "Expected at least one citation"
assert all(c.type == "page_location" for c in pdf_sources)

Now the same question against the **plain text** of the report. This time each citation gives character positions, so we can **prove** every quote is really in the report by slicing the text ourselves.

In [ ]:
text_document = {
    "type": "document",
    "title": "halvora_report.md",
    "source": {"type": "text", "media_type": "text/plain", "data": report_text},
    "citations": {"enabled": True},
}

response = track(client.messages.create(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": [text_document, {"type": "text", "text": question}]}],
))
text_sources = render_cited_answer(response)

print("Checking every citation against the report:")
for n, c in enumerate(text_sources, 1):
    exact = report_text[c.start_char_index:c.end_char_index] == c.cited_text
    print(f"   [{n}] report_text[{c.start_char_index}:{c.end_char_index}] matches the quote? {'yes' if exact else 'NO'}")

assert text_sources, "Expected at least one citation"
assert all(report_text[c.start_char_index:c.end_char_index] == c.cited_text for c in text_sources)

**What to notice**

- Every `[n]` points to a real quote, and for plain text we just proved it in code. A UI could use those positions to highlight the source sentence.
- **PDF gives page numbers, text gives character positions.** Pick the document type that gives your readers the reference they need.
- This is the API's version of the *"Sources:"* line we asked for in a prompt in the RAG notebooks. Here the quotes are exact, and Claude can't paraphrase them.

## 5. Prompt caching: many questions, one PDF (paid)

In the board Q&A you'll ask a dozen questions about the **same** PDF. Without caching, every question sends and processes those ~13,000 tokens again, at full price.

**How it works:** mark a content block with `"cache_control": {"type": "ephemeral"}`. The API stores the prompt **up to and including that block**. The next request that starts with **exactly the same content** reads it from the cache instead.

### The rules

| Rule | Detail |
|---|---|
| **Matches from the start** | The prompt is read in this order: `tools`, then `system`, then `messages`. Change one character early on, and everything after it misses the cache |
| **Stable content first** | The document goes first and the changing question after it. Anything that changes must come **after** the marked block |
| **Up to 4 markers** | Each `cache_control` marks one breakpoint |
| **A minimum size** | On Sonnet 5.5 the cached part must be at least **512 tokens**. Shorter prompts are silently not cached, with no error, so check the usage numbers |
| **Lifetime** | 5 minutes, refreshed for free on every hit. A 1-hour option exists: `{"type": "ephemeral", "ttl": "1h"}`, at a higher write price |
| **Price on Sonnet 5.5** | Writing costs $2.50 per million tokens, 1.25 times normal input. **Reading costs $0.10**, just 5% of normal input |

**Reading the usage line:** `input` is processed at full price, `cache_write` is stored in the cache, and `cache_read` comes from the cache.

In [ ]:
def ask_board(question, system=None):
    params = {
        "model": MODEL,
        "max_tokens": 300,
        "output_config": {"effort": "low"},
        "messages": [{"role": "user", "content": [
            pdf_block(cache=True),                          # stable, so cached (the breakpoint is here)
            {"type": "text", "text": question},             # changes every time, so it goes after
        ]}],
    }
    if system:
        params["system"] = system
    return track(client.messages.create(**params))


board_questions = [
    "How much did revenue grow in 2023? One sentence.",
    "How many people did Halvora hire, and where did it open an office? One sentence.",
    "What is XDR-47? One sentence.",
    "What are the three priorities for 2024? One sentence.",
]

cache_log = []
for question in board_questions:
    response = ask_board(question)
    cache_log.append(response.usage)
    print(f"Q: {question}\nA: {answer_text(response)}\n")

assert any((usage.cache_read_input_tokens or 0) > 0 for usage in cache_log), "Expected at least one cache read"

In [ ]:
# What did caching save? Compare with sending the PDF at full price every time.
with_cache = sum(u.input_tokens * INPUT_PRICE + (u.cache_creation_input_tokens or 0) * CACHE_WRITE_PRICE
                 + (u.cache_read_input_tokens or 0) * CACHE_READ_PRICE for u in cache_log) / 1_000_000
without_cache = sum((u.input_tokens + (u.cache_creation_input_tokens or 0) + (u.cache_read_input_tokens or 0))
                    * INPUT_PRICE for u in cache_log) / 1_000_000

print(f"{'call':<6}{'input':>8}{'cache_write':>13}{'cache_read':>12}")
for i, u in enumerate(cache_log, 1):
    print(f"{i:<6}{u.input_tokens:>8,}{u.cache_creation_input_tokens or 0:>13,}{u.cache_read_input_tokens or 0:>12,}")

print(f"\nInput cost with caching   : ${with_cache:.4f}")
print(f"Input cost without caching: ${without_cache:.4f}")
print(f"Caching saved {1 - with_cache / without_cache:.0%} on input for these {len(cache_log)} questions.")

**What to notice**

- **The first question may already be a cache read.** Section 3 sent the same PDF block, with no system prompt, a moment ago, so it wrote the cache. If more than 5 minutes passed, the first question writes it again instead.
- After that, the PDF comes from the cache, and only the short question is processed at full price.
- The more questions you ask within 5 minutes, the bigger the saving.

### What breaks the cache?

The system prompt comes **before** the PDF in the prompt. So adding one changes the start of the prompt, and the PDF has to be written to the cache again.

In [ ]:
response = ask_board("Who presented the Q4 security incident report? One sentence.",
                     system="You are preparing Halvora's board briefing.")
usage = response.usage
print("With a new system prompt:", "cache READ" if usage.cache_read_input_tokens else "cache MISS, written again",
      f"(write={usage.cache_creation_input_tokens or 0:,}, read={usage.cache_read_input_tokens or 0:,})")

**What to notice**

- Any change **before** the marked block counts: the tools list, the system prompt, earlier messages, even thinking settings. Keep those stable for as long as you want cache hits.
- A miss isn't an error. It just costs more, so watch `cache_write` and `cache_read`.

## 6. All together: the board brief (paid)

One request now combines everything:

- the **PDF**, with **citations** and **caching**,
- the **chart image**,
- **adaptive thinking**, to plan five strong bullets before writing them.

It uses a new system prompt and new thinking settings, so it writes the cache again rather than reading it. Run it twice within 5 minutes and the second run reads the cache.

In [ ]:
start = time.perf_counter()
brief = track(client.messages.create(
    model=MODEL,
    max_tokens=6000,
    thinking={"type": "adaptive", "display": "summarized"},
    output_config={"effort": "medium"},
    system="You write concise, factual board briefings. Every claim must come from the attached report.",
    messages=[{"role": "user", "content": [
        pdf_block(citations=True, cache=True),
        image_block,
        {"type": "text", "text": "Write a 5-bullet board brief on Halvora's 2023: finance, engineering, the "
                                 "security incident, people, and 2024 priorities. One sentence per bullet. "
                                 "Use the chart to sanity-check the engineering numbers."},
    ]}],
))
print(f"Took {time.perf_counter() - start:.1f}s | block types: {sorted({block.type for block in brief.content})}")
brief_sources = render_cited_answer(brief)

assert brief_sources, "Expected the brief to cite the report"

**What to notice**

- Each feature is just a field or a content block: `thinking`, `output_config`, the document block with `citations` and `cache_control`, and the image block. They combine freely, apart from a few documented exceptions, such as citations with structured outputs.
- Every bullet should carry citations back to report pages, so a board member can check any claim.

## Final spend

In [ ]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

## Takeaways

| Feature | The rule to remember |
|---|---|
| Thinking | Adaptive by default on Sonnet 5.5. Control it with `effort`, and turn it off with `between_tools`. You pay for thinking even when you only see a summary |
| Images | Put the image block before the question. It costs about ⌈width ÷ 28⌉ × ⌈height ÷ 28⌉ tokens. Check any numbers Claude reads |
| PDFs | Claude gets text plus an image of every page, so charts work. It costs several times the plain text |
| Citations | `citations: {"enabled": True}` gives exact quotes with pages or character positions. Not combinable with structured outputs |
| Prompt caching | Stable content first, `cache_control` on it, then check `cache_write` and `cache_read`. Writes cost 1.25 times input, and reads 5% on Sonnet 5.5, for 5 minutes |

## Your turn

- **About 2 cents:** in section 1, add `"adaptive, medium effort"` to `SETTINGS`. Where does it land between low and high?
- **About 1 cent:** in section 2, delete the two `ax.bar_label(...)` lines, re-draw the chart, and re-run. Can Claude still read exact values from the bar heights alone?
- **About 2 cents:** in section 4, ask a question the report can't answer, such as *"What is the CEO's salary?"*. Do any citations appear?
- **About 1 cent:** in section 5, add four more board questions. How does the saving change?

**Next:** [10 · Model Context Protocol](10_model_context_protocol.ipynb): packaging tools, data and prompts in a server that any app can use.